# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path(r"C:\Machine_Learning_worksheet\Data\cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
-> user_id 로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
-> 기존 게이트 위치인 gate_30 을 대조군, 게이트를 40단계로 옮긴 gate_40을 실험군으로 설정

- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
핵심 지표는 단순 1일 평가보다는 7일 이후 시점을 나타내는 retention_7로 
보조지표는 초기 반응 확인을 위해 retention 1로
가드레일 지표는 활동이 감소하는지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수 )로 보겠다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?



In [ ]:
# 여기에 코드를 작성하세요.


---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
-> 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻고 코드를 확인할 수 있게 하기 위해서 

- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
-> user_id별로 practice_group의 고유값 수를 계산하여 최대값이 1인지 확인한다.


- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
-> p-value 가 0.05 보다 작으므로, 50:50 계획과 통계적으로 일치 한다고 보기 어렵다.


- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
세 변수는 버전을 경험한 이후의 측정된 결과이므로 처치영향을 받을 수 있다.
균형 점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인 같은 공변량이 필요하지만 현 데이터에서는 제공하지 않았다.


In [4]:
# 여기에 코드를 작성하세요.

print(f"Users: {len(df):,} | Duplicates: {df['userid'].duplicated().sum()} | Missing: {df.isna().sum().sum()}")
print(df["version"].value_counts(normalize=True).round(4))
print(f"rounds max: {df['sum_gamerounds'].max():,} / median: {df['sum_gamerounds'].median():.0f}")

metrics = df.groupby("version").agg(
    users=("userid", "count"),
    retention_1=("retention_1", "mean"),
    retention_7=("retention_7", "mean"),
    mean_rounds=("sum_gamerounds", "mean"),
)
print(metrics.round(4))


Users: 90,189 | Duplicates: 0 | Missing: 0
version
gate_40    0.5044
gate_30    0.4956
Name: proportion, dtype: float64
rounds max: 49,854 / median: 16
         users  retention_1  retention_7  mean_rounds
version                                              
gate_30  44700       0.4482       0.1902      52.4563
gate_40  45489       0.4423       0.1820      51.2988


---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   h0 : gate 30과 gate 40의 7-day retentiona rate은 동일하다.
   h1 : gate 30 과 gate 40 의 7-day retention rate 은 다르다

   유의 수준 : 0.05
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.

   unit of randomization            user
   control                          gate(30)
   treatment                        gate(40)
   allocation                        50:50
   primary metric                   7 day retention rate
   secondary metric                 1day retention rate, game rounds
   guardrail                        crash rate, 결제전환율

3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
   - SRM: 배정비율이 50:50에서 벗어나면 랜덤화 로직 결함 의심
   - Duplicate user_id: 중복 시 독립성이 깨지고 효과가 희석됨
   - Outlier: sum_gamerounds의 봇·테스트 계정 의심값 확인

4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
   -Spillover: 친구 초대 기능으로 정보 공유 발생 가능
   - 다중기기: 같은 사람이 양쪽 그룹에 포함될 가능성

버전별 retention 집계 → two-proportion z-test → 신뢰구간·효과 크기 확인 
→ guardrail 점검 후 배포 판단

5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.
7-day retention은 gate_30 19.02%, gate_40 18.20%로 0.82%p 차이였고 1-day retention도 같은 방향이었다. 배정 비율은 49.56:50.44로 SRM 문제가 없었고 중복·결측치도 없었다. 다만 sum_gamerounds는 중앙값 16에 최댓값 49,854로 극단값이 있어 평균 해석에 주의가 필요하다. 이 수치는 관측된 차이일 뿐 우연인지 실제 효과인지는 아직 검정하지 않아 알 수 없다. 따라서 gate_40 배포 결정은 보류하고, two-proportion z-test로 유의성과 신뢰구간을 확인한 뒤 guardrail을 점검해 최종 판단해야 한다.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [2]:
# 여기에 코드를 작성하세요.

# ---------- Data Quality Checks ----------
print("===== Data Quality =====")
print(f"Total users: {len(df):,}")
print(f"Duplicate userid: {df['userid'].duplicated().sum()}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"\nGroup sizes (SRM check):")
print(df["version"].value_counts())

# Sample Ratio Mismatch — 50:50 배정에서 크게 벗어나면 배정 로직 의심
counts = df["version"].value_counts()
ratio = counts / counts.sum()
print(f"Split ratio: {ratio.round(4).to_dict()}")

# Outliers in engagement
print(f"\nsum_gamerounds max: {df['sum_gamerounds'].max():,}")
print(f"sum_gamerounds median: {df['sum_gamerounds'].median():.0f}")
print(f"Users with 0 rounds: {(df['sum_gamerounds'] == 0).sum():,} "
      f"({(df['sum_gamerounds'] == 0).mean():.2%})")

# ---------- Metrics by Version ----------
metrics = df.groupby("version").agg(
    users=("userid", "count"),
    retention_1=("retention_1", "mean"),
    retention_7=("retention_7", "mean"),
    mean_rounds=("sum_gamerounds", "mean"),
    median_rounds=("sum_gamerounds", "median"),
)

print("\n===== Metrics by Version =====")
print(metrics.round(4))

# 차이 (아직 검정 아님 — 단순 관측값)
for m in ["retention_1", "retention_7", "mean_rounds"]:
    diff = metrics.loc["gate_40", m] - metrics.loc["gate_30", m]
    print(f"{m}: gate_40 − gate_30 = {diff:+.4f}")

===== Data Quality =====
Total users: 90,189
Duplicate userid: 0
Missing values: 0

Group sizes (SRM check):
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
Split ratio: {'gate_40': 0.5044, 'gate_30': 0.4956}

sum_gamerounds max: 49,854
sum_gamerounds median: 16
Users with 0 rounds: 3,994 (4.43%)

===== Metrics by Version =====
         users  retention_1  retention_7  mean_rounds  median_rounds
version                                                             
gate_30  44700       0.4482       0.1902      52.4563           17.0
gate_40  45489       0.4423       0.1820      51.2988           16.0
retention_1: gate_40 − gate_30 = -0.0059
retention_7: gate_40 − gate_30 = -0.0082
mean_rounds: gate_40 − gate_30 = -1.1575


---

## 실습 마무리

- 어떤 문제가 있었는가?
그룹별 유지율과 평균 게임 라운드만 비교하여 사용자의 고정 배정 여부, 계획한 표본 비율, 지표 사전 지정, 외부간섭, 로깅 등의 문제를 놓칠 수 있다.
- 어떻게 개선했는가?
실험 단위를 userid로 명시하고 대조군, 실험군과 3가지 지표를 사전에 정의 했다.
실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과 변수, 사전 공변량의 차이 등을 함께 점검 했다.


- 무엇을 근거로 개선되었다고 판단했는가?
사용자 중복 0건, 실제 굽비율 약 49.5% 50.4%, 적합도 검정 p-value가 0.05미만을 확인하여 단순히 비슷해 보인다는 필수 1의 파단보다는 구체적인 조사 증거를 확보 했다.
40 gate를 배포전에 기술 검정을 통해 버전 업 베포 결정을 보류 했다.



단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
